## Настройки Colab

In [1]:
# Настройка пользователя (сделать один раз)
!git config --global user.email "nabludatellip@gmail.com"
!git config --global user.name "ProninPV"

In [3]:
from getpass import getpass

# 1. Безопасный ввод токена
GITHUB_TOKEN = getpass('Введите ваш GitHub Personal Access Token: ')

Введите ваш GitHub Personal Access Token: ··········


In [4]:
!git clone https://github.com/ProninPV/ml-regression_concrete-strength.git
%cd ml-regression_concrete-strength

Cloning into 'ml-regression_concrete-strength'...
remote: Enumerating objects: 611, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 611 (delta 20), reused 24 (delta 16), pack-reused 575 (from 1)
Receiving objects: 100% (611/611), 32.08 MiB | 21.00 MiB/s, done.
Resolving deltas: 100% (324/324), done.
/content/ml-regression_concrete-strength


In [5]:
%cd /content/ml-regression_concrete-strength
!git config pull.rebase false
!git pull origin submit_1

/content/ml-regression_concrete-strength
From https://github.com/ProninPV/ml-regression_concrete-strength
 * branch            submit_1   -> FETCH_HEAD
Updating 89655eb..7a413eb
Fast-forward
 catboost_info/catboost_training.json               | 1962 ++++++-
 catboost_info/learn/events.out.tfevents            |  Bin 4798 -> 91010 bytes
 catboost_info/learn_error.tsv                      | 1960 ++++++-
 catboost_info/time_left.tsv                        | 1960 ++++++-
 config/config.yaml                                 |  378 +-
 data/processed/data_train_outliers.csv             |  771 ---
 data/processed/data_train_outliers.parquet         |  Bin 23834 -> 0 bytes
 data/processed/data_train_outliers.pkl             |  Bin 68740 -> 0 bytes
 data/processed/y_train_outliers.pkl                |  Bin 19103 -> 0 bytes
 models/modeling_report/modeling_experiments.csv    |  111 -
 .../modeling_experiments_20251115_135319.csv       |   11 -
 .../modeling_experiments_20251115_140340.csv       | 

In [6]:
!git branch

* tuning


In [7]:
!git checkout submit_1

Branch 'submit_1' set up to track remote branch 'submit_1' from 'origin'.
Switched to a new branch 'submit_1'


In [8]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.9 MB/s eta 0:00:00


In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 6.2 MB/s eta 0:00:00


## 6.0 Импорты библиотек

In [13]:
import os
import yaml
import logging
import pickle
import numpy as np
import scipy.stats as stats
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import levene
from scipy.stats import ttest_ind
from typing import List, Any, Optional, Tuple, Dict, Union
from datetime import datetime
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import cross_val_score, KFold, RepeatedKFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import VotingRegressor, StackingRegressor
from sklearn.compose import ColumnTransformer
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import time
import psutil
from tqdm import tqdm
import gc
import optuna
import joblib

In [12]:
!pip install -U lightautoml

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.5/412.5 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.1/216.1 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.5/309.5 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.6/223.6 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.2/201.2 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 104.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 8.3 MB/s eta 0:00:00
  Created wheel for json2html: filename=json2html-1.3.0-py3-none-any.whl size=7591 sha256=c5aec73d0b6cd3cc06d787988f88a652dde963acf9a23ca1240bacd32f5c6e37
  Stor

In [30]:
from lightautoml.automl.presets.tabular_presets import TabularAutoML, TabularUtilizedAutoML
from lightautoml.dataset.roles import DatetimeRole
from lightautoml.tasks import Task
from lightautoml.report.report_deco import ReportDeco

In [14]:
import warnings
warnings.filterwarnings("ignore")

In [15]:
# расширяем поле ноутбука для удобства
from IPython.display import display, HTML
display(HTML('<style>.container {width:87% !important;}</style>'))
display(HTML("<style>.output_scroll {height:auto !important; max-height:10000px !important;}</style>"))

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [16]:
# Настройки для pandas (количество отображаемых колонок)
pd.set_option('display.max_columns', 100)

In [17]:
# Определение стиля для pyplot
plt.style.use('ggplot')

In [18]:
# В Colab проект клонируется в /content/
# Устанавливаем правильную рабочую директорию
project_root = Path('/content/ml-regression_concrete-strength')

# Определяем корень проекта
# cwd = Path().resolve()
# project_root = cwd.parent

# Добавляем корень проекта в sys.path (этого достаточно)
sys.path.append(str(project_root))

# Проверяем наличие конфиг файла
config_path = project_root / "config" / "config.yaml"
print(f"Looking for config at: {config_path}")

# Загрузка данных из config.yaml
from src.data import downloader, loader, preprocessor, saving
from src.features import feat_preprocessing
from src.modeling import modeling

# Передаем путь явно
config = loader.load_config(config_path)
print("✅ Config loaded successfully!")

Looking for config at: /content/ml-regression_concrete-strength/config/config.yaml
✅ Config loaded successfully!


## 6.1. Загрузка данных

In [19]:
# Загрузка train
df_train = loader.data_load_preprocessed(data_type='train',
                                         config=config)

[⧗] Загружаю данные из: /content/ml-regression_concrete-strength/data/processed/eda_data_train.pkl
[✓] Данные успешно загружены. Форма: (781, 11)


In [20]:
# Вывод первых 5 строк тренировочного датасета
df_train.head()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age,Strength,W/C,Sp/C_pct
0,376.0,0.0,0.0,214.6,0.0,1003.5,762.4,3,16.28,0.570745,0.000000
1,491.0,26.0,123.0,210.0,3.9,882.0,699.0,56,59.59,0.427699,0.007943
2,250.0,0.0,95.7,187.4,5.5,956.9,861.2,3,13.82,0.749600,0.022000
3,310.0,0.0,0.0,192.0,0.0,1012.0,830.0,90,35.76,0.619355,0.000000
4,252.1,97.1,75.6,193.8,8.3,835.5,821.4,28,33.40,0.768743,0.032923


## 6.2. Обучение LightAutoML

In [24]:
task = Task('reg')

In [25]:
target = 'Strength'
rs = 42

In [26]:
roles = {
    'target': target
}

In [31]:
timeout = 3600
threads = 4
cv = 5

In [38]:
%%time
automl = TabularAutoML(task = task,
                       timeout = timeout,
                       cpu_limit = threads,
                       reader_params = {'n_jobs': threads, 'random_state': rs, 'cv': cv})


RD = ReportDeco(output_path = 'tabularAutoML_model_report')

automl_rd = RD(
    automl
)

oof_pred = automl_rd.fit_predict(df_train, roles = roles, verbose = -1)

Выходные данные были обрезаны до нескольких последних строк (5000).
INFO2:lightautoml.ml_algo.base:===== Start working with fold 1 for Lvl_0_Pipe_0_Mod_0_LinearL2 =====
INFO3:lightautoml.ml_algo.torch_based.linear_model:Linear model: C = 1e-05 score = -262.9408663112711
INFO3:lightautoml.ml_algo.torch_based.linear_model:Linear model: C = 5e-05 score = -220.33954902952334
INFO3:lightautoml.ml_algo.torch_based.linear_model:Linear model: C = 0.0001 score = -191.47163696274706
INFO3:lightautoml.ml_algo.torch_based.linear_model:Linear model: C = 0.0005 score = -133.90510769558807
INFO3:lightautoml.ml_algo.torch_based.linear_model:Linear model: C = 0.001 score = -121.47451846741326
INFO3:lightautoml.ml_algo.torch_based.linear_model:Linear model: C = 0.005 score = -112.41476613542244
INFO3:lightautoml.ml_algo.torch_based.linear_model:Linear model: C = 0.01 score = -111.23082783495298
INFO3:lightautoml.ml_algo.torch_based.linear_model:Linear model: C = 0.05 score = -109.44044438030727
INFO3:li

CPU times: user 8min 48s, sys: 39.4 s, total: 9min 27s
Wall time: 6min 13s


In [39]:
# Извлекаем значения
y_true = df_train['Strength'].values  # реальные значения
y_pred = oof_pred.data[:, 0]  # предсказанные значения (первый столбец)

# Считаем RMSE
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
print(f"RMSE: {rmse:.4f}")

RMSE: 4.2432


## Отправка на Github


In [40]:
!git status

On branch submit_1
Your branch is up to date with 'origin/submit_1'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	tabularAutoML_model_report/

nothing added to commit but untracked files present (use "git add" to track)


In [41]:
# 1. Добавляем файлы
!git add .

# 2. Коммитим
!git commit -m "fit: add tuning XGboost regression"

[submit_1 9439575] fit: add tuning XGboost regression
 5 files changed, 393 insertions(+)
 create mode 100644 tabularAutoML_model_report/feature_importance.png
 create mode 100644 tabularAutoML_model_report/lama_interactive_report.html
 create mode 100644 tabularAutoML_model_report/valid_error_hist.png
 create mode 100644 tabularAutoML_model_report/valid_scatter_plot.png
 create mode 100644 tabularAutoML_model_report/valid_target_distribution.png


In [42]:
!git push https://{GITHUB_TOKEN}@github.com/ProninPV/ml-regression_concrete-strength.git submit_1

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (8/8), done.
Writing objects: 100% (8/8), 350.42 KiB | 23.36 MiB/s, done.
Total 8 (delta 1), reused 4 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/ProninPV/ml-regression_concrete-strength.git
   7a413eb..9439575  submit_1 -> submit_1
